# LC9 — Optimisation: how much can Svedala actually carry? (hands-on, ~45 min)

In Lab 2 your screener proved the Svedala snapshot violates N-1 — and you wrote down a guess: *what fraction of that load can the grid serve securely?* Today we answer it, and learn the standard computational tool of power systems on the way. In EG2130 you found limits by **stepping** until something broke. The engineer's move is to **formulate**: variables, objective, constraints — and let a solver find the binding constraint for you. Material from this notebook appears in **Quiz 4**.

## 1. The formulation, on paper first

> **maximize** k  (the load scaling factor)
> **over** k and the generator dispatch p₁…p₃₆
> **subject to**
> - power balance: Σ pᵢ = k · (total load)
> - every line flow within its limit: |fₗ| ≤ fₗᵐᵃˣ
> - every generator within its range: 0 ≤ pᵢ ≤ pᵢᵐᵃˣ

The flows are the awkward part — they depend on all injections at once. The **DC power flow** approximation makes them *linear*: f = PTDF · (injections), where the PTDF matrix says how one MW injected at bus j loads line ℓ. Linear objective, linear constraints → a **linear program**, solvable to global optimality in milliseconds.

In [ ]:
# numba is optional but makes pandapower ~10x faster — this notebook runs
# a few hundred power flows, so it is worth the install.
%pip install pandapower numba scipy numpy pandas --quiet

In [ ]:
from pathlib import Path
# Guard: this notebook expects to run from the notebooks/ folder of a clone of
# the course repository — the datasets live one level up in ../data/.
# Failing here, early and clearly, beats a confusing FileNotFoundError later.
assert Path("../data").exists(), (
    "Course data folder not found. Clone KTH-EG2140/course-material and open "
    "this notebook from its notebooks/ folder.")

In [ ]:
import numpy as np
import pandas as pd
import pandapower as pp

# the toolbox loader's logic, inline (your svedala-toolbox has this as a module)
def load_svedala(d="../data/svedala"):
    b=pd.read_csv(f"{d}/buses.csv",index_col=0); l=pd.read_csv(f"{d}/lines.csv",index_col=0)
    t=pd.read_csv(f"{d}/transformers.csv",index_col=0); g=pd.read_csv(f"{d}/generators.csv",index_col=0)
    ld=pd.read_csv(f"{d}/loads.csv",index_col=0)
    net=pp.create_empty_network(); D={400.0:2.0,220.0:1.0,135.0:0.9}
    for i,r in b.iterrows(): pp.create_bus(net,vn_kv=r.vn_kv,name=r["name"],zone=r.SubGeographicalRegion_name,in_service=r.in_service,index=i)
    for i,r in l.iterrows():
        ika=r.max_i_ka if pd.notna(r.max_i_ka) else D.get(net.bus.at[r.from_bus,"vn_kv"],0.5)
        pp.create_line_from_parameters(net,from_bus=r.from_bus,to_bus=r.to_bus,length_km=r.length_km,
            r_ohm_per_km=r.r_ohm_per_km,x_ohm_per_km=r.x_ohm_per_km,c_nf_per_km=r.c_nf_per_km,
            max_i_ka=ika,name=r["name"],in_service=r.in_service,index=i)
    for i,r in t.iterrows():
        pp.create_transformer_from_parameters(net,hv_bus=r.hv_bus,lv_bus=r.lv_bus,sn_mva=r.sn_mva,
            vn_hv_kv=r.vn_hv_kv,vn_lv_kv=r.vn_lv_kv,vk_percent=r.vk_percent,vkr_percent=r.vkr_percent,
            pfe_kw=r.pfe_kw,i0_percent=r.i0_percent,shift_degree=r.shift_degree,name=r["name"],in_service=r.in_service,index=i)
    for i,r in g.iterrows():
        pp.create_gen(net,bus=r.bus,p_mw=r.p_mw,vm_pu=r.vm_pu,sn_mva=r.sn_mva,min_q_mvar=r.min_q_mvar,
            max_q_mvar=r.max_q_mvar,slack=r.slack,name=r["name"],in_service=r.in_service,index=i)
    for i,r in ld.iterrows():
        pp.create_load(net,bus=r.bus,p_mw=r.p_mw,q_mvar=r.q_mvar,name=r["name"],in_service=r.in_service,index=i)
    return net

net = load_svedala()
P_LOAD = net.load.p_mw.sum()
print(f"{len(net.bus)} buses | base load {P_LOAD:.0f} MW")

## 2. The PTDF matrix — sensitivities by experiment

We *measure* the PTDF numerically: run a DC power flow, inject 1 extra MW at a bus (withdrawn at the slack), and record how every line flow changes. One column per bus:

In [ ]:
slack_bus = net.gen.loc[net.gen.slack, "bus"].iloc[0]
buses = list(net.bus.index); lines = list(net.line.index)

def dc_flows():
    pp.rundcpp(net)
    return net.res_line.p_from_mw.values.copy()

base_flow = dc_flows()
PTDF = np.zeros((len(lines), len(buses)))
# One temporary "probe" load that we move from bus to bus. Setting p_mw = -1
# means injecting 1 MW; the slack absorbs it. Column j of the PTDF is then
# simply (flows with the probe at bus j) minus (base flows).
probe = pp.create_load(net, bus=slack_bus, p_mw=0.0)
for j, b in enumerate(buses):
    net.load.at[probe, "bus"] = b; net.load.at[probe, "p_mw"] = -1.0   # inject 1 MW
    PTDF[:, j] = dc_flows() - base_flow
# Remove the probe again — the network is back to its base state.
net.load.at[probe, "p_mw"] = 0.0
net.load.drop(probe, inplace=True)
print("PTDF shape:", PTDF.shape, "| e.g. 1 MW at bus", buses[10], "loads line RL1 by",
      f"{PTDF[0,10]:+.3f} MW")

## 3. Line limits in MW, and the LP

DC works in MW, so convert each line's current limit: fᵐᵃˣ ≈ √3 · V · Iᵐᵃˣ. Then hand the whole formulation to `scipy.optimize.linprog` — variables x = [k, p₁…p₃₆]:

In [ ]:
from scipy.optimize import linprog
V = net.bus.loc[net.line.from_bus, "vn_kv"].values
F_MAX = np.sqrt(3) * V * net.line.max_i_ka.values          # MW (approx, cosphi=1)

load_bus = np.zeros(len(buses)); gen_bus = {}
for _, r in net.load.iterrows():
    load_bus[buses.index(r.bus)] += r.p_mw / P_LOAD         # load share per bus
G = list(net.gen.index)
A_gen = np.zeros((len(buses), len(G)))
for gi, g in enumerate(G):
    A_gen[buses.index(net.gen.at[g, "bus"]), gi] = 1.0

# injections(x) = A_gen @ p  -  k * P_LOAD * load_bus      (per bus, MW)
# flows(x)     = base-independent: PTDF @ injections
n = 1 + len(G)
c = np.zeros(n); c[0] = -1.0                               # maximize k
A_flow = np.hstack([(-P_LOAD * PTDF @ load_bus)[:, None], PTDF @ A_gen])
A_ub = np.vstack([A_flow, -A_flow])
b_ub = np.concatenate([F_MAX, F_MAX])
A_eq = np.zeros((1, n)); A_eq[0, 0] = -P_LOAD; A_eq[0, 1:] = 1.0
b_eq = [0.0]
bounds = [(0, None)] + [(0, 1.2 * net.gen.at[g, "p_mw"]) for g in G]  # 20% redispatch headroom

res = linprog(c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method="highs")
k_n0 = res.x[0]
print(f"N-0 optimum: k = {k_n0:.3f}  ->  {k_n0*P_LOAD:.0f} MW servable with redispatch")
binding = np.where(np.isclose(A_flow @ res.x, F_MAX, rtol=0.01))[0]
print("binding lines:", [net.line.at[lines[i], 'name'] for i in binding[:5]])

Read that result the way an engineer does: the solver did not *search* — it went straight to the binding constraints and told you *which lines* set the limit. Shadow prices (the dual values) would even tell you what one more MW of capacity on each is worth — that is where market clearing and nodal prices come from, same machinery, bigger scale.

## 4. The reality check: your screener is the judge

The LP is DC, N-0, and cos φ = 1. The Lab 2 question was about **N-1, AC** security. So: scale the real network, run the real screener, bisect:

In [ ]:
def n1_secure(k):
    """AC N-1 check at load scaling k: rebuild, scale loads and (proportionally)
    generation, run the base case, then every single-line outage — the Lab 2
    screener's logic. True only if everything converges within line limits."""
    m = load_svedala()
    m.load.p_mw *= k; m.load.q_mvar *= k
    m.gen.loc[~m.gen.slack, "p_mw"] *= k
    try: pp.runpp(m)
    except Exception: return False
    if not m.converged or m.res_line.loading_percent.max() > 100: return False
    for li in m.line.index:
        m.line.at[li, "in_service"] = False
        try:
            pp.runpp(m, init="results")
            ok = m.converged and m.res_line.loading_percent.max() <= 100
        except Exception:
            ok = False
        finally:
            m.line.at[li, "in_service"] = True
        if not ok: return False
    return True

# Bisection on k: n1_secure() is monotone in k (more load never makes the
# system safer), so 7 halvings pin the boundary to about 0.5% of k.
lo, hi = 0.3, 1.0
for _ in range(7):
    mid = (lo + hi) / 2
    lo, hi = (mid, hi) if n1_secure(mid) else (lo, mid)
k_n1 = lo
print(f"N-1 secure limit (AC, proportional dispatch): k ≈ {k_n1:.2f}"
      f"  ->  {k_n1*P_LOAD:.0f} MW")
print(f"Compare: N-0 DC optimum k = {k_n0:.2f}. Security costs "
      f"{(k_n0-k_n1)*P_LOAD:.0f} MW of servable load.")

**There is your Lab 2 answer** — check it against the guess in your repo. And notice the gap between the two numbers: that distance is what *security-constrained* OPF (the LP with N-1 flow constraints built in — same idea, ~52× more constraints) exists to close professionally. TSOs solve exactly that, continent-sized, every day.

## 5. Where this machinery lives in practice

- **OPF / SCOPF**: what you just did, with AC or N-1 constraints — operations planning.
- **Unit commitment**: add on/off decisions → mixed-integer LP — day-ahead scheduling.
- **Market clearing**: maximize welfare instead of load → the day-ahead price *is* a shadow price.

One formulation pattern, most of the industry's decisions.

## Self-check

In [ ]:
assert 0.45 < k_n1 < 0.65, "N-1 secure limit outside expected band"
assert k_n0 > k_n1, "redispatch N-0 optimum should exceed the proportional N-1 limit"
assert res.status == 0
print(f"ALL OK — Svedala serves ~{k_n1:.0%} of the snapshot load N-1 securely; "
      "the snapshot itself is a stressed planning case, as your screener told you in Lab 2.")